# Tech Pulse — Incremental Data Updater

Checks what is already stored in GitHub for every data source and fetches only
what is missing. Run this instead of the individual collector notebooks.

## Sources covered
| # | Source | Location | Update strategy |
|---|--------|----------|-----------------|
| 4 | Hacker News | `Raw Data Pulled/hn_data/` | Re-fetch the current month |
| 5 | Reddit | `Raw Data Pulled/reddit_data/` | Re-fetch current month per subreddit |
| 6 | GDELT | `Raw Data Pulled/gdelt_data/` | Re-fetch the last 30 days per ticker |
| 7 | News sentiment | `Raw Data Pulled/stocktwits_data/` | Alpha Vantage + Polygon latest articles |
| 8 | SEC EDGAR | `Raw Data Pulled/edgar_data/` | Filings newer than the stored maximum |
| 9 | Stock prices | `stock_tracking/stocks/` | From each ticker's last stored date |

All paths match the current repository layout: raw collections live under
`Raw Data Pulled/`, and prices live under `stock_tracking/`.

**Estimated runtime:** 30–60 minutes for a full pass.

---
## 0. Install

In [1]:
# !pip install requests pandas python-dotenv yfinance

---
## 1. Configuration

In [2]:
import os, io, time, base64, requests, warnings, calendar
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO  = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', None)

# Repository layout
RAW_FOLDER   = 'Raw Data Pulled'
TRACK_FOLDER = 'stock_tracking'

HN_PREFIX         = RAW_FOLDER   + '/hn_data'
REDDIT_PREFIX     = RAW_FOLDER   + '/reddit_data'
GDELT_PREFIX      = RAW_FOLDER   + '/gdelt_data'
STOCKTWITS_PREFIX = RAW_FOLDER   + '/stocktwits_data'
EDGAR_PREFIX      = RAW_FOLDER   + '/edgar_data'
STOCKS_PREFIX     = TRACK_FOLDER + '/stocks'

# News sentiment API keys (Section 7). StockTwits itself is behind Cloudflare,
# so these two providers stand in for it.
ALPHA_VANTAGE_KEY = os.environ.get('ALPHA_VANTAGE_KEY', 'YOUR_KEY_HERE')
POLYGON_KEY       = os.environ.get('POLYGON_KEY',       'YOUR_KEY_HERE')

SUBREDDITS = ['wallstreetbets', 'stocks', 'investing', 'technology', 'SecurityAnalysis']

GDELT_COMPANIES = {
    'AAPL': 'Apple Inc',        'MSFT': 'Microsoft',      'GOOGL': 'Google',
    'META': 'Meta Platforms',   'AMZN': 'Amazon',         'NVDA' : 'Nvidia',
    'TSM' : 'TSMC',             'INTC': 'Intel Corporation',
    'AMD' : 'Advanced Micro Devices',                     'QCOM' : 'Qualcomm',
    'TSLA': 'Tesla',            'COIN': 'Coinbase',       'PYPL' : 'PayPal',
    'NFLX': 'Netflix',          'CRM' : 'Salesforce',     'CRWD' : 'CrowdStrike',
    'PANW': 'Palo Alto Networks',                         'PLTR' : 'Palantir',
    'DDOG': 'Datadog',          'SNOW': 'Snowflake Inc',  'MDB'  : 'MongoDB',
    'NOW' : 'ServiceNow',       'OKTA': 'Okta Inc',       'NVO'  : 'Novo Nordisk',
    'INCY': 'Incyte Corporation',                         'KGC'  : 'Kinross Gold',
    'PM'  : 'Philip Morris',    'WPM' : 'Wheaton Precious Metals',
}

NEWS_TICKERS = list(GDELT_COMPANIES.keys()) + ['SPY', 'QQQ']

STOCK_TICKERS = None   # None = read the ticker list from stocks/index.csv

NOW       = datetime.now(timezone.utc)
TODAY     = NOW.strftime('%Y-%m-%d')
CUR_YEAR  = NOW.year
CUR_MONTH = NOW.month

print('Incremental updater configured')
print(f'  Repo   : {GITHUB_REPO}')
print(f'  Token  : {"set" if GITHUB_TOKEN else "NOT SET"}')
print(f'  Today  : {TODAY}')
print(f'  Raw    : {RAW_FOLDER}/')
print(f'  Prices : {STOCKS_PREFIX}/')
print(f'  AlphaV : {"set" if ALPHA_VANTAGE_KEY != "YOUR_KEY_HERE" else "not set"}')
print(f'  Polygon: {"set" if POLYGON_KEY != "YOUR_KEY_HERE" else "not set"}')

Incremental updater configured
  Repo   : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Token  : NOT SET
  Today  : 2026-07-29
  Raw    : Raw Data Pulled/
  Prices : stock_tracking/stocks/
  AlphaV : not set
  Polygon: not set


---
## 2. GitHub helpers

In [3]:
GITHUB_API = 'https://api.github.com'

def _gh_headers(token):
    return {'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28'}

def load_csv_raw(path, token=None):
    """Read a CSV from the repo. Spaces in folder names are URL-encoded."""
    safe = path.replace(' ', '%20')
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{safe}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404:
        raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content:
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

def push_csv(df, path, token, msg=None):
    """Write a CSV back to the repo, creating or updating as needed."""
    if msg is None:
        msg = f'Update {path} - {len(df):,} rows [{TODAY}]'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = _gh_headers(token)
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha:
        payload['sha'] = sha
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'    saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha:
                payload['sha'] = sha
        else:
            print(f'    FAILED {path}: {resp.status_code}')
            return False

print('GitHub helpers loaded')

GitHub helpers loaded


---
## 3. Status Check

Run this first to see how stale each source is before committing to a full pass.

In [4]:
print('Checking what is already stored ...\n')
status = {}

def _safe_latest(path, col, token):
    try:
        df = load_csv_raw(path, token)
        if df.empty or col not in df.columns:
            return None, 0
        return str(pd.to_datetime(df[col], errors='coerce').max().date()), len(df)
    except FileNotFoundError:
        return None, 0
    except Exception:
        return None, 0

# HN
try:
    idx = load_csv_raw(f'{HN_PREFIX}/hn_index.csv', GITHUB_TOKEN)
    status['HN'] = (str(idx['date_max'].max()), int(idx['story_count'].sum()))
except Exception:
    status['HN'] = (None, 0)

# Reddit
try:
    idx = load_csv_raw(f'{REDDIT_PREFIX}/reddit_index.csv', GITHUB_TOKEN)
    status['Reddit'] = (str(idx['date_max'].max()), int(idx['post_count'].sum()))
except Exception:
    status['Reddit'] = (None, 0)

# GDELT — sample one ticker for recency
status['GDELT'] = _safe_latest(f'{GDELT_PREFIX}/gdelt_NVDA_{CUR_YEAR}.csv', 'date', GITHUB_TOKEN)

# News sentiment
try:
    idx = load_csv_raw(f'{STOCKTWITS_PREFIX}/stocktwits_index.csv', GITHUB_TOKEN)
    status['News sentiment'] = (str(idx['date_max'].max()), int(idx['post_count'].sum()))
except Exception:
    status['News sentiment'] = (None, 0)

# EDGAR
status['EDGAR'] = _safe_latest(f'{EDGAR_PREFIX}/edgar_AAPL.csv', 'date', GITHUB_TOKEN)

# Stock prices
status['Stock prices'] = _safe_latest(f'{STOCKS_PREFIX}/prices_SPY.csv', 'Date', GITHUB_TOKEN)

print(f'{"Source":<18} {"Latest date":<14} {"Rows":>12}   Status')
print('-' * 62)
for name, (latest, rows) in status.items():
    if latest is None:
        print(f'{name:<18} {"no data":<14} {rows:>12,}   needs a full collection')
        continue
    days = (pd.Timestamp(TODAY) - pd.Timestamp(latest)).days
    state = 'current' if days <= 1 else f'{days} days behind'
    print(f'{name:<18} {latest:<14} {rows:>12,}   {state}')

print(f'\nToday: {TODAY}')
print('Run sections 4-9 to update each source.')

Checking what is already stored ...

Source             Latest date            Rows   Status
--------------------------------------------------------------
HN                 2026-07-10          937,777   19 days behind
Reddit             2026-07-10          910,969   19 days behind
GDELT              2026-07-08            1,250   21 days behind
News sentiment     no data                   0   needs a full collection
EDGAR              2026-05-01              359   89 days behind
Stock prices       2026-07-02            4,149   27 days behind

Today: 2026-07-29
Run sections 4-9 to update each source.


---
## 4. Hacker News

Re-fetches the current month plus any months missing from this year.

In [5]:
HN_BASE       = 'https://hn.algolia.com/api/v1/search_by_date'
HITS_PER_PAGE = 1000
HN_MIN_POINTS = 3
HN_DELAY      = 0.25

def month_bounds_unix(year, month):
    start = int(datetime(year, month, 1, tzinfo=timezone.utc).timestamp())
    if month == 12:
        nxt = datetime(year + 1, 1, 1, tzinfo=timezone.utc)
    else:
        nxt = datetime(year, month + 1, 1, tzinfo=timezone.utc)
    return start, int(nxt.timestamp()) - 1

def fetch_hn_month(year, month):
    ts_from, ts_to = month_bounds_unix(year, month)
    params = {'tags': 'story', 'hitsPerPage': HITS_PER_PAGE,
              'numericFilters': f'created_at_i>={ts_from},created_at_i<={ts_to},points>={HN_MIN_POINTS}',
              'page': 0}
    hits, page = [], 0
    while True:
        params['page'] = page
        resp = requests.get(HN_BASE, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get('hits', [])
        hits.extend(batch)
        if page >= data.get('nbPages', 1) - 1 or not batch:
            break
        page += 1
        time.sleep(HN_DELAY)
    rows = []
    for h in hits:
        created = h.get('created_at', '')
        try:
            dt = pd.to_datetime(created, utc=True)
        except Exception:
            dt = None
        rows.append({'id': h.get('objectID'), 'title': h.get('title'),
                     'url': h.get('url', ''), 'author': h.get('author'),
                     'points': h.get('points'), 'num_comments': h.get('num_comments'),
                     'created_at': created, 'created_at_i': h.get('created_at_i'),
                     'tags': ', '.join(h.get('_tags', [])),
                     'year': dt.year if dt is not None else None,
                     'month': dt.month if dt is not None else None,
                     'day': dt.day if dt is not None else None,
                     'date': str(dt.date()) if dt is not None else None,
                     'day_of_week': dt.day_name() if dt is not None else None})
    return pd.DataFrame(rows)

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    print(f'Hacker News: updating {CUR_YEAR} ...')
    year_path = f'{HN_PREFIX}/hn_{CUR_YEAR}.csv'
    try:
        df_existing = load_csv_raw(year_path, GITHUB_TOKEN)
        done_months = set(pd.to_numeric(df_existing['month'], errors='coerce').dropna().astype(int))
    except FileNotFoundError:
        df_existing, done_months = pd.DataFrame(), set()

    frames = [df_existing] if not df_existing.empty else []
    for month in range(1, CUR_MONTH + 1):
        if month < CUR_MONTH and month in done_months:
            continue
        df_m = fetch_hn_month(CUR_YEAR, month)
        if not df_m.empty:
            frames.append(df_m)
            print(f'  {calendar.month_abbr[month]}: {len(df_m):,} stories')
        time.sleep(HN_DELAY)

    if frames:
        df_year = (pd.concat(frames, ignore_index=True)
                   .drop_duplicates(subset='id')
                   .sort_values('created_at_i')
                   .reset_index(drop=True))
        push_csv(df_year, year_path, GITHUB_TOKEN, f'HN {CUR_YEAR}: {len(df_year):,} stories')

        try:
            df_idx = load_csv_raw(f'{HN_PREFIX}/hn_index.csv', GITHUB_TOKEN)
            df_idx = df_idx[df_idx['year'] != CUR_YEAR]
        except FileNotFoundError:
            df_idx = pd.DataFrame()
        new_row = pd.DataFrame([{'year': CUR_YEAR, 'story_count': len(df_year),
                                 'months_done': df_year['month'].nunique(),
                                 'date_min': df_year['date'].min(),
                                 'date_max': df_year['date'].max()}])
        df_idx = pd.concat([df_idx, new_row], ignore_index=True).sort_values('year')
        push_csv(df_idx, f'{HN_PREFIX}/hn_index.csv', GITHUB_TOKEN, 'Update HN index')
    print('Hacker News complete')

GITHUB_TOKEN not set


---
## 5. Reddit

Re-fetches the current month for each subreddit.

In [6]:
ARCTIC_BASE  = 'https://arctic-shift.photon-reddit.com/api'
REDDIT_BATCH = 100
REDDIT_DELAY = 1.0

def fetch_reddit_month(subreddit, year, month):
    start_ts = datetime(year, month, 1, tzinfo=timezone.utc).timestamp()
    last_day = calendar.monthrange(year, month)[1]
    end_ts   = datetime(year, month, last_day, 23, 59, 59, tzinfo=timezone.utc).timestamp()
    posts, cursor = [], start_ts
    while cursor < end_ts:
        params = {'subreddit': subreddit, 'after': int(cursor), 'before': int(end_ts),
                  'limit': REDDIT_BATCH, 'sort': 'asc',
                  'fields': 'id,author,created_utc,subreddit,title,selftext,score,num_comments'}
        resp = requests.get(f'{ARCTIC_BASE}/posts/search', params=params, timeout=30)
        if resp.status_code != 200:
            break
        batch = resp.json().get('data', [])
        if not batch:
            break
        posts.extend(batch)
        cursor = batch[-1]['created_utc'] + 1
        if len(batch) < REDDIT_BATCH:
            break
        time.sleep(REDDIT_DELAY)
    rows = []
    for p in posts:
        created = p.get('created_utc', 0)
        dt = datetime.fromtimestamp(created, tz=timezone.utc)
        rows.append({'id': p.get('id'), 'subreddit': subreddit, 'author': p.get('author'),
                     'title': p.get('title', ''), 'selftext': (p.get('selftext') or '')[:500],
                     'score': p.get('score', 0), 'num_comments': p.get('num_comments', 0),
                     'created_utc': created, 'date': str(dt.date()),
                     'year': dt.year, 'month': dt.month, 'day': dt.day,
                     'day_of_week': dt.strftime('%A')})
    return pd.DataFrame(rows)

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    print(f'Reddit: updating {CUR_YEAR} ...')
    year_path = f'{REDDIT_PREFIX}/reddit_{CUR_YEAR}.csv'
    try:
        df_existing = load_csv_raw(year_path, GITHUB_TOKEN)
        done = set(zip(df_existing['subreddit'],
                       pd.to_numeric(df_existing['month'], errors='coerce').fillna(0).astype(int)))
    except FileNotFoundError:
        df_existing, done = pd.DataFrame(), set()

    frames = [df_existing] if not df_existing.empty else []
    for subreddit in SUBREDDITS:
        for month in range(1, CUR_MONTH + 1):
            if month < CUR_MONTH and (subreddit, month) in done:
                continue
            df_m = fetch_reddit_month(subreddit, CUR_YEAR, month)
            if not df_m.empty:
                frames.append(df_m)
                print(f'  r/{subreddit} {calendar.month_abbr[month]}: {len(df_m):,} posts')
            time.sleep(REDDIT_DELAY)

    if frames:
        df_year = (pd.concat(frames, ignore_index=True)
                   .drop_duplicates(subset='id')
                   .sort_values('created_utc')
                   .reset_index(drop=True))
        push_csv(df_year, year_path, GITHUB_TOKEN, f'Reddit {CUR_YEAR}: {len(df_year):,} posts')

        try:
            df_idx = load_csv_raw(f'{REDDIT_PREFIX}/reddit_index.csv', GITHUB_TOKEN)
            df_idx = df_idx[df_idx['year'] != CUR_YEAR]
        except FileNotFoundError:
            df_idx = pd.DataFrame()
        new_row = pd.DataFrame([{'year': CUR_YEAR, 'post_count': len(df_year),
                                 'date_min': df_year['date'].min(),
                                 'date_max': df_year['date'].max()}])
        df_idx = pd.concat([df_idx, new_row], ignore_index=True).sort_values('year')
        push_csv(df_idx, f'{REDDIT_PREFIX}/reddit_index.csv', GITHUB_TOKEN, 'Update Reddit index')
    print('Reddit complete')

GITHUB_TOKEN not set


---
## 6. GDELT

Re-fetches the last 30 days per ticker.

Note: GDELT's `artlist` mode does not return a usable tone value, so every row
carries `tone = 0`. Notebook A now scores the article titles with VADER instead,
which is why the `title` column matters here.

In [7]:
GDELT_DOC_API = 'https://api.gdeltproject.org/api/v2/doc/doc'
GDELT_DELAY   = 5.0

def fetch_gdelt_range(query, start_date, end_date, max_records=250):
    params = {'query': f'"{query}"', 'mode': 'artlist', 'format': 'json',
              'startdatetime': start_date.replace('-', '') + '000000',
              'enddatetime':   end_date.replace('-', '')   + '235959',
              'maxrecords': max_records, 'sort': 'DateDesc'}
    for attempt in range(3):
        try:
            resp = requests.get(GDELT_DOC_API, params=params, timeout=60)
            if resp.status_code == 200:
                return resp.json().get('articles', [])
            return []
        except Exception:
            wait = (attempt + 1) * 10
            print(f'    timeout, retrying in {wait}s ...')
            time.sleep(wait)
    return []

def gdelt_to_df(articles, ticker):
    if not articles:
        return pd.DataFrame()
    rows = []
    for a in articles:
        seen = a.get('seendate', '')
        try:
            dt = datetime.strptime(seen[:8], '%Y%m%d').replace(tzinfo=timezone.utc)
        except Exception:
            dt = None
        tone = a.get('tone', '')
        try:
            tone_val = float(str(tone).split(',')[0]) if tone else 0.0
        except Exception:
            tone_val = 0.0
        rows.append({'ticker': ticker, 'date': str(dt.date()) if dt else None,
                     'title': a.get('title', ''), 'url': a.get('url', ''),
                     'domain': a.get('domain', ''), 'language': a.get('language', ''),
                     'tone': round(tone_val, 4), 'source': 'gdelt_gkg'})
    return pd.DataFrame(rows)

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    fetch_from = (NOW - timedelta(days=30)).strftime('%Y-%m-%d')
    print(f'GDELT: fetching {fetch_from} to {TODAY} for {len(GDELT_COMPANIES)} tickers ...')
    for ticker, name in GDELT_COMPANIES.items():
        year_path = f'{GDELT_PREFIX}/gdelt_{ticker}_{CUR_YEAR}.csv'
        try:
            df_existing = load_csv_raw(year_path, GITHUB_TOKEN)
        except FileNotFoundError:
            df_existing = pd.DataFrame()

        df_new = gdelt_to_df(fetch_gdelt_range(name, fetch_from, TODAY), ticker)
        if df_new.empty:
            print(f'  {ticker}: nothing new')
            time.sleep(GDELT_DELAY)
            continue

        if not df_existing.empty:
            df_combined = (pd.concat([df_existing, df_new], ignore_index=True)
                           .drop_duplicates(subset=['date', 'url'])
                           .sort_values('date')
                           .reset_index(drop=True))
        else:
            df_combined = df_new
        push_csv(df_combined, year_path, GITHUB_TOKEN,
                 f'GDELT {ticker} {CUR_YEAR}: +{len(df_new)} articles')
        print(f'  {ticker}: +{len(df_new)} articles')
        time.sleep(GDELT_DELAY)
    print('GDELT complete')

GITHUB_TOKEN not set


---
## 7. News Sentiment (Alpha Vantage + Polygon)

This is the source that replaced StockTwits after it went behind Cloudflare.
It writes to `stocktwits_data/` so that Notebook A keeps finding it under the
name it already expects.

**This folder is currently missing from the repository** — it was dropped during
the reorganisation, which is why the source contributes nothing at the moment.
Running this section recreates it.

Set `ALPHA_VANTAGE_KEY` and `POLYGON_KEY` in Section 1 or in your `.env` file.
Both free tiers allow 5 requests per minute, hence the delays.

In [8]:
AV_DELAY      = 12.5
POLY_DELAY    = 13.0
AV_ENABLED    = ALPHA_VANTAGE_KEY != 'YOUR_KEY_HERE'
POLY_ENABLED  = POLYGON_KEY       != 'YOUR_KEY_HERE'

def fetch_alphavantage_news(ticker, limit=200):
    params = {'function': 'NEWS_SENTIMENT', 'tickers': ticker,
              'limit': limit, 'apikey': ALPHA_VANTAGE_KEY}
    try:
        resp = requests.get('https://www.alphavantage.co/query', params=params, timeout=30)
        if resp.status_code != 200:
            return []
        payload = resp.json()
        if 'Note' in payload or 'Information' in payload:
            print('    Alpha Vantage rate limit reached')
            return []
        return payload.get('feed', [])
    except Exception as e:
        print(f'    Alpha Vantage error: {e}')
        return []

def av_to_df(articles, ticker):
    rows = []
    for a in articles:
        ticker_sent, relevance = 0.0, 0.0
        for ts in a.get('ticker_sentiment', []):
            if ts.get('ticker') == ticker:
                ticker_sent = float(ts.get('ticker_sentiment_score', 0) or 0)
                relevance   = float(ts.get('relevance_score', 0) or 0)
                break
        try:
            dt = pd.to_datetime(a.get('time_published', ''), format='%Y%m%dT%H%M%S')
        except Exception:
            dt = None
        rows.append({'id': a.get('url', '')[-60:], 'ticker': ticker,
                     'title': a.get('title', ''), 'body': (a.get('summary') or '')[:500],
                     'source': a.get('source', ''), 'url': a.get('url', ''),
                     'created_at': str(dt) if dt is not None else '',
                     'date': str(dt.date()) if dt is not None else None,
                     'year': dt.year if dt is not None else None,
                     'month': dt.month if dt is not None else None,
                     'overall_sentiment': float(a.get('overall_sentiment_score', 0) or 0),
                     'ticker_sentiment': ticker_sent, 'relevance_score': relevance,
                     'st_sentiment': 'Bullish' if ticker_sent > 0.15 else ('Bearish' if ticker_sent < -0.15 else ''),
                     'provider': 'alpha_vantage'})
    return pd.DataFrame(rows)

def fetch_polygon_news(ticker, limit=50):
    params = {'ticker': ticker, 'limit': limit, 'order': 'desc', 'apiKey': POLYGON_KEY}
    try:
        resp = requests.get('https://api.polygon.io/v2/reference/news', params=params, timeout=30)
        if resp.status_code != 200:
            return []
        return resp.json().get('results', [])
    except Exception as e:
        print(f'    Polygon error: {e}')
        return []

def poly_to_df(news, ticker):
    rows = []
    for n in news:
        dt_str = n.get('published_utc', '')
        try:
            dt = pd.to_datetime(dt_str)
        except Exception:
            dt = None
        rows.append({'id': n.get('id', ''), 'ticker': ticker,
                     'title': n.get('title', ''), 'body': (n.get('description') or '')[:500],
                     'source': n.get('publisher', {}).get('name', ''),
                     'url': n.get('article_url', ''), 'created_at': dt_str,
                     'date': str(dt.date()) if dt is not None else None,
                     'year': dt.year if dt is not None else None,
                     'month': dt.month if dt is not None else None,
                     'overall_sentiment': 0.0, 'ticker_sentiment': 0.0,
                     'relevance_score': 1.0, 'st_sentiment': '',
                     'provider': 'polygon'})
    return pd.DataFrame(rows)

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
elif not AV_ENABLED and not POLY_ENABLED:
    print('No API keys set. Add ALPHA_VANTAGE_KEY and/or POLYGON_KEY in Section 1.')
else:
    print(f'News sentiment: {len(NEWS_TICKERS)} tickers '
          f'(Alpha Vantage {"on" if AV_ENABLED else "off"}, '
          f'Polygon {"on" if POLY_ENABLED else "off"}) ...')
    index_rows = []
    for i, ticker in enumerate(NEWS_TICKERS, 1):
        path = f'{STOCKTWITS_PREFIX}/stocktwits_{ticker}.csv'
        try:
            df_existing = load_csv_raw(path, GITHUB_TOKEN)
        except FileNotFoundError:
            df_existing = pd.DataFrame()

        frames = []
        if AV_ENABLED:
            arts = fetch_alphavantage_news(ticker)
            if arts:
                frames.append(av_to_df(arts, ticker))
            time.sleep(AV_DELAY)
        if POLY_ENABLED:
            news = fetch_polygon_news(ticker)
            if news:
                frames.append(poly_to_df(news, ticker))
            time.sleep(POLY_DELAY)

        frames = [f for f in frames if not f.empty]
        if not frames:
            print(f'  [{i}/{len(NEWS_TICKERS)}] {ticker}: nothing returned')
            continue

        df_new = pd.concat(frames, ignore_index=True).drop_duplicates(subset='id')
        if not df_existing.empty:
            df_combined = (pd.concat([df_existing, df_new], ignore_index=True)
                           .drop_duplicates(subset='id')
                           .sort_values('date', ascending=False)
                           .reset_index(drop=True))
        else:
            df_combined = df_new.sort_values('date', ascending=False).reset_index(drop=True)

        push_csv(df_combined, path, GITHUB_TOKEN,
                 f'News sentiment {ticker}: {len(df_combined):,} articles')
        print(f'  [{i}/{len(NEWS_TICKERS)}] {ticker}: +{len(df_new)} new, {len(df_combined):,} total')

        index_rows.append({'ticker': ticker, 'post_count': len(df_combined),
                           'date_min': df_combined['date'].min(),
                           'date_max': df_combined['date'].max(),
                           'bullish': int((df_combined['st_sentiment'] == 'Bullish').sum()),
                           'bearish': int((df_combined['st_sentiment'] == 'Bearish').sum())})

    if index_rows:
        df_idx = pd.DataFrame(index_rows).sort_values('ticker')
        push_csv(df_idx, f'{STOCKTWITS_PREFIX}/stocktwits_index.csv', GITHUB_TOKEN,
                 f'News sentiment index: {int(df_idx["post_count"].sum()):,} articles')
        print(f'\nNews sentiment complete: {int(df_idx["post_count"].sum()):,} articles')
        print(df_idx.to_string(index=False))

GITHUB_TOKEN not set


---
## 8. SEC EDGAR

Adds filings newer than the latest date already stored per ticker.

In [9]:
EDGAR_DELAY  = 0.15
SEC_HEADERS  = {'User-Agent': 'TechPulse research contact@example.com'}  # put a real address here
FILING_TYPES = ['8-K', '10-Q', '10-K', '13-F', 'S-1']

def get_recent_filings(cik, filing_types):
    resp = requests.get(f'https://data.sec.gov/submissions/CIK{cik}.json',
                        headers=SEC_HEADERS, timeout=30)
    if resp.status_code != 200:
        return []
    recent = resp.json().get('filings', {}).get('recent', {})
    forms  = recent.get('form', [])
    dates  = recent.get('filingDate', [])
    accs   = recent.get('accessionNumber', [])
    docs   = recent.get('primaryDocument', [])
    out = []
    for i, form in enumerate(forms):
        if form in filing_types:
            out.append({'form': form,
                        'date': dates[i] if i < len(dates) else None,
                        'accession': accs[i] if i < len(accs) else None,
                        'document': docs[i] if i < len(docs) else None})
    return out

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    print('EDGAR: loading CIK lookup ...')
    resp = requests.get('https://www.sec.gov/files/company_tickers.json',
                        headers=SEC_HEADERS, timeout=20)
    cik_data = resp.json() if resp.status_code == 200 else {}
    wanted = set(GDELT_COMPANIES.keys())
    ticker_to_cik = {}
    for entry in cik_data.values():
        t = str(entry.get('ticker', '')).upper()
        if t in wanted:
            ticker_to_cik[t] = str(entry['cik_str']).zfill(10)
    print(f'  CIKs resolved: {len(ticker_to_cik)}/{len(wanted)}')

    updated = 0
    for ticker in sorted(wanted):
        cik = ticker_to_cik.get(ticker)
        if not cik:
            continue
        path = f'{EDGAR_PREFIX}/edgar_{ticker}.csv'
        try:
            df_existing = load_csv_raw(path, GITHUB_TOKEN)
            latest_date = str(df_existing['date'].max())
        except FileNotFoundError:
            df_existing, latest_date = pd.DataFrame(), '1993-01-01'

        filings = get_recent_filings(cik, FILING_TYPES)
        time.sleep(EDGAR_DELAY)
        if not filings:
            continue

        df_new = pd.DataFrame(filings)
        df_new['ticker'] = ticker
        df_new['cik']    = cik
        df_new = df_new[df_new['date'] > latest_date]
        if df_new.empty:
            continue

        df_combined = (pd.concat([df_existing, df_new], ignore_index=True)
                       .drop_duplicates(subset=['accession'])
                       .sort_values('date', ascending=False)
                       .reset_index(drop=True))
        push_csv(df_combined, path, GITHUB_TOKEN, f'EDGAR {ticker}: +{len(df_new)} filings')
        print(f'  {ticker}: +{len(df_new)} filings')
        updated += 1
    print(f'EDGAR complete: {updated} tickers updated')

GITHUB_TOKEN not set


---
## 9. Stock Prices

Fetches from each ticker's last stored date to today.

In [10]:
try:
    import yfinance as yf
    YF_AVAILABLE = True
except ImportError:
    YF_AVAILABLE = False
    print('yfinance missing: pip install yfinance')

if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
elif not YF_AVAILABLE:
    print('Install yfinance first')
else:
    if STOCK_TICKERS is None:
        try:
            df_stk_idx = load_csv_raw(f'{STOCKS_PREFIX}/index.csv', GITHUB_TOKEN)
            tickers = [t for t in df_stk_idx['ticker'].dropna().astype(str) if t.lower() != 'nan']
            print(f'Loaded {len(tickers):,} tickers from index.csv')
        except FileNotFoundError:
            tickers = list(GDELT_COMPANIES.keys()) + ['SPY', 'QQQ']
            print(f'No index found, defaulting to {len(tickers)} tickers')
    else:
        tickers = STOCK_TICKERS

    updated, current, failed = 0, 0, []
    for i, ticker in enumerate(tickers, 1):
        path = f'{STOCKS_PREFIX}/prices_{ticker}.csv'
        try:
            df_existing = load_csv_raw(path, GITHUB_TOKEN)
            df_existing['Date'] = pd.to_datetime(df_existing['Date'])
            fetch_from = (df_existing['Date'].max() + timedelta(days=1)).strftime('%Y-%m-%d')
            if fetch_from >= TODAY:
                current += 1
                if i % 100 == 0:
                    print(f'  [{i}/{len(tickers)}] {current} already current ...')
                continue
        except FileNotFoundError:
            df_existing, fetch_from = pd.DataFrame(), '2010-01-01'

        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                df_new = yf.download(ticker, start=fetch_from, end=TODAY,
                                     progress=False, auto_adjust=True)
            if df_new.empty:
                current += 1
                continue
            if isinstance(df_new.columns, pd.MultiIndex):
                df_new.columns = df_new.columns.get_level_values(0)
            df_new = df_new.reset_index()
            df_new['ticker'] = ticker
            if not df_existing.empty:
                df_combined = (pd.concat([df_existing, df_new], ignore_index=True)
                               .drop_duplicates(subset='Date')
                               .sort_values('Date')
                               .reset_index(drop=True))
            else:
                df_combined = df_new
            push_csv(df_combined, path, GITHUB_TOKEN,
                     f'{ticker}: +{len(df_new)} rows through {TODAY}')
            print(f'  [{i}/{len(tickers)}] {ticker}: +{len(df_new)} rows')
            updated += 1
            time.sleep(0.3)
        except Exception:
            failed.append(ticker)

    print(f'\nStock prices complete')
    print(f'  Updated         : {updated}')
    print(f'  Already current : {current}')
    print(f'  Failed          : {len(failed)} {failed[:10] if failed else ""}')

GITHUB_TOKEN not set


---
## 10. Final Status Check

Confirm every source is current, then re-run notebooks A, B and C.

In [11]:
print('Post-update status\n')
checks = [
    ('HN',             f'{HN_PREFIX}/hn_index.csv',                 'date_max'),
    ('Reddit',         f'{REDDIT_PREFIX}/reddit_index.csv',         'date_max'),
    ('GDELT',          f'{GDELT_PREFIX}/gdelt_NVDA_{CUR_YEAR}.csv', 'date'),
    ('News sentiment', f'{STOCKTWITS_PREFIX}/stocktwits_index.csv', 'date_max'),
    ('EDGAR',          f'{EDGAR_PREFIX}/edgar_AAPL.csv',            'date'),
    ('Stock prices',   f'{STOCKS_PREFIX}/prices_SPY.csv',           'Date'),
]
print(f'{"Source":<18} {"Latest":<14} Status')
print('-' * 50)
for name, path, col in checks:
    try:
        df = load_csv_raw(path, GITHUB_TOKEN)
        latest = pd.to_datetime(df[col], errors='coerce').max()
        if pd.isna(latest):
            print(f'{name:<18} {"no dates":<14} check the file')
            continue
        days = (pd.Timestamp(TODAY) - latest).days
        print(f'{name:<18} {latest.strftime("%Y-%m-%d"):<14} '
              f'{"current" if days <= 2 else str(days) + " days behind"}')
    except FileNotFoundError:
        print(f'{name:<18} {"missing":<14} run that section above')
    except Exception as e:
        print(f'{name:<18} {"error":<14} {str(e)[:30]}')

print('\nNext: re-run A_sentiment_engine, then B_correlation_engine, then C_strategy_engine.')

Post-update status

Source             Latest         Status
--------------------------------------------------
HN                 2026-07-10     19 days behind
Reddit             2026-07-10     19 days behind
GDELT              2026-07-08     21 days behind
News sentiment     missing        run that section above
EDGAR              2026-05-01     89 days behind
Stock prices       2026-07-02     27 days behind

Next: re-run A_sentiment_engine, then B_correlation_engine, then C_strategy_engine.
